# Cell-Type Resolution: HuBMAP

**Estimated time:** 25 minutes

Move from bulk heart tissue to ventricular cardiac myocytes. Compare mean
normalized expression with the percentage of records above zero.


## Cell-type expression

GTEx showed expression in bulk heart tissue, which contains many cell types.
HuBMAP lets us examine records assigned to a selected heart cell type.

HuBMAP stands for **Human BioMolecular Atlas Program**. This NIH Common Fund
program maps cells and molecules within human tissues. Its Cells API connects
aggregated pipeline outputs and indexed expression values to labels such as
ventricular cardiac myocyte, fibroblast, or macrophage.


### Why ventricular cardiac myocytes?

The [source paper](https://doi.org/10.1038/s41598-025-88465-8) studied
early-onset advanced heart failure. The highest yield of pathogenic or likely
pathogenic variants occurred in hypertrophic, dilated, and arrhythmogenic right
ventricular cardiomyopathy. Sarcomeric variants were most common in the first
two groups, while desmosomal variants were concentrated in the third. The
teaching table includes *MYH7*, *MYBPC3*, *TNNT2*, *ACTC1*, *DSG2*, *DSC2*,
and *PKP2*.

The paper did not analyze cell types, so HuBMAP adds cell-type information rather than
reproducing the study. We begin with ventricular cardiac myocytes because the
ventricles provide the heart's main pumping force and many candidate genes
contribute to cardiac-muscle contraction or structure. This choice also follows
the GTEx left-ventricle comparison.

The request retrieves up to the first 500 indexed records labeled as regular
ventricular cardiac myocytes. The fixed limit keeps the live request
manageable.

Learn more about the resource's APIs in the
[HuBMAP API documentation](https://docs.hubmapconsortium.org/apis.html).


### How to read the returned values

The available samples, cell labels, and processed datasets determine which
genes return values and which remain unavailable.

**Why HuBMAP helps:**
GTEx combines expression from all cell types in a tissue sample. HuBMAP allows
us to ask a more focused question: which candidate genes have indexed values in
ventricular cardiac myocytes? For the genes with data, we will compare their
mean normalized expression with how often those values are above zero.


## Querying the HuBMAP API

The Cells API intersects heart and ventricular cardiac-myocyte records, retrieves
up to the first 500 records, and requests one gene at a time. A gene with no
indexed value is unavailable, not zero.


### Prepare the gene list and API helper

Load the published variants, select the 25 unique gene symbols, and import the
HuBMAP wrapper.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Locate the repository root.
REPO_ROOT = Path.cwd() if Path("api_helpers.py").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Import the API wrapper.
from api_helpers import fetch_hubmap_ventricular_context

# Load the published variants.
DATA_DIR = REPO_ROOT / "data"
variants = pd.read_csv(DATA_DIR / "variants.csv")
gene_symbols = sorted(variants["gene_symbol"].unique())
pd.Series(gene_symbols, name="gene_symbol").head()


`gene_symbols` contains the 25 unique gene symbols in alphabetical order.
`variants` contains all 54 published rows.


### Request ventricular cardiac-myocyte expression

Request ventricular cardiac-myocyte expression for the 25 genes. The wrapper
queries one gene at a time, so this request may take longer than the GTEx
request. Start by checking `availability`, then compare expression only among
genes with returned measurements.


In [ ]:
# Query the 25 genes.
hubmap = fetch_hubmap_ventricular_context(gene_symbols)

# If the live request is unavailable, use the dated teaching response instead.
# hubmap = pd.read_csv(DATA_DIR / "hubmap_cell_expression.csv")
# hubmap = hubmap[
#     hubmap["cell_type_id"] == "CL:0002131"
# ].reset_index(drop=True)

# Inspect expression and availability.
hubmap.head()


One row represents one queried gene. `mean_normalized_expression` is the mean
normalized expression across retrieved records, and `percent_detected` is the
percentage of those records with a value above zero. The separate `availability` field states whether the index returned
any values.

**Live data:**
Values are requested directly from HuBMAP. Coverage and service availability
can change.


### How the wrapper works

The wrapper follows five steps:

1. Create a query handle for heart records.
2. Create a query handle for regular ventricular cardiac myocytes
   (`CL:0002131`).
3. Intersect the two sets.
4. Retrieve up to the first 500 records for each gene.
5. Calculate mean normalized expression and the percentage of values above zero.

The implementation and its support functions are in
[`api_helpers.py`](https://cfdetrainingcenter.github.io/candidate-genetic-variants/api_helpers.py).


## Review ventricular cardiac myocytes

We can now ask which genes have data, their mean normalized expression, and
how often those values are above zero.


### Summarize data availability

Keep the ventricular cardiac-myocyte rows and count the genes with and without
indexed expression values. Start with coverage before comparing measurements.


In [ ]:
# Select ventricular cardiac myocytes.
ventricular = hubmap[
    hubmap["cell_type_id"] == "CL:0002131"
].copy()

# Count genes by data availability.
availability_summary = (
    ventricular["availability"]
    .value_counts()
    .rename_axis("availability")
    .reset_index(name="genes")
)
availability_summary


In the dated teaching data, 14 genes have indexed ventricular cardiac-myocyte
values and 11 are unavailable. Compare measurements only among the 14 genes
with data.


### Compare available genes

Display all available genes in descending order of mean normalized expression.
The table includes `percent_detected` so you can compare average values with the
percentage of retrieved records above zero.


In [ ]:
# Keep genes with returned values.
available_ventricular = ventricular[
    ventricular["availability"] == "available"
]

# Rank all available genes by mean normalized expression.
mean_ranked_genes = (
    available_ventricular
    .sort_values("mean_normalized_expression", ascending=False)
    .set_index("gene_symbol")
)
mean_ranked_genes.loc[
    :, ["mean_normalized_expression", "percent_detected"]
]


In the dated teaching data, *MYH7* and *DES* have the highest mean normalized
expression. Their values are above zero in 39.8% and 35.2% of the retrieved records.
The mean summarizes expression across the retrieved records, while the
percentage detected reports how many records had a value above zero.


### Compare detection frequency

Rank the available genes by percent detected and compare the result with the
mean-expression ranking.


In [ ]:
# Rank genes by detection frequency.
top_detected_genes = (
    available_ventricular
    .nlargest(10, "percent_detected")
    .set_index("gene_symbol")
)
top_detected_genes.loc[
    :, ["mean_normalized_expression", "percent_detected"]
]


The change in ranking makes the distinction visible. In the dated teaching
data, *TNNT2* ranks second by percent detected but sixth by mean expression.
*DES* ranks second by mean expression and third by percent detected. Neither
summary replaces the other because they describe different expression patterns.


### Plot ventricular expression

Plot mean normalized expression for the selected genes. Interpret the chart
together with the percentage-above-zero table and availability summary.


In [ ]:
# Keep the 10 highest means for a readable plot.
top_ventricular_genes = mean_ranked_genes.head(10)

# Plot mean normalized expression.
axis = top_ventricular_genes["mean_normalized_expression"].plot.bar(
    color="#3d64b3",
    figsize=(10, 5),
)
axis.set_ylabel("Mean normalized expression")
axis.set_xlabel("Gene from the source paper")
axis.set_title("HuBMAP expression in ventricular cardiac myocytes")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


The plot shows the 10 highest means from the complete 14-gene table. It is a
focused view, not the complete result. The 11 unavailable genes remain visible
in the availability summary rather than disappearing from the analysis.


## Quiz yourself!
What does high mean normalized expression with a lower `percent_detected` indicate?

- Larger returned values occur in a smaller portion of the retrieved records
- Every retrieved record has the same expression value
- The gene is unavailable in the Cells API index

<details>
<summary>Show answer and feedback</summary>

- **Larger returned values occur in a smaller portion of the retrieved records:** Correct. The mean summarizes expression across the retrieved records, while `percent_detected` reports the percentage with a value above zero.
- **Every retrieved record has the same expression value:** A mean and a percentage above zero do not show that every record has the same value.
- **The gene is unavailable in the Cells API index:** An unavailable gene has no returned values and is excluded from this comparison.

</details>

What does an unavailable HuBMAP value mean here?

- The expression index did not provide a value
- The gene was not expressed

<details>
<summary>Show answer and feedback</summary>

- **The expression index did not provide a value:** Correct. Keep this result separate from a returned value of zero.
- **The gene was not expressed:** Choose unavailable because the index returned no value for this gene.

</details>


## Key points

- HuBMAP connects indexed expression values with cell-type annotations from
  processed atlas data.
- Ventricular cardiac myocytes provide a focused cell type for studying the
  cardiomyopathy phenotypes and cardiac-muscle genes reported in the paper.
- The wrapper calculates mean normalized expression and the percentage of returned
  records above zero. These summaries can produce different gene rankings.
- An unavailable indexed value and a returned value of zero are recorded
  separately.

**Next:** Use Pharos to examine what is known about the encoded proteins and
which research tools are available.
